In [1]:
import pandas as pd
import numpy as np
import torch

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

In [2]:
FEATURES_PATH = "../datasets/Elliptic_Bitcoin/elliptic_txs_features.csv"
CLASSES_PATH = "../datasets/Elliptic_Bitcoin/elliptic_txs_classes.csv"
EDGES_PATH = "../datasets/Elliptic_Bitcoin/elliptic_txs_edgelist.csv"

In [3]:
features_df = pd.read_csv(
    FEATURES_PATH,
    header=None
)

classes_df = pd.read_csv(
    CLASSES_PATH
)

edges_df = pd.read_csv(
    EDGES_PATH
)

In [4]:
print("Features shape:", features_df.shape)
print("Classes shape :", classes_df.shape)
print("Edges shape   :", edges_df.shape)

Features shape: (203769, 167)
Classes shape : (203769, 2)
Edges shape   : (234355, 2)


In [5]:
display(features_df.head())
display(classes_df.head())
display(edges_df.head())

,0,1,2,3,4,5,6,7,8,9,...,157,158,159,160,161,162,163,164,165,166
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117


,txId,class
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown
3,232438397,2
4,230460314,unknown


,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206


In [6]:
num_columns = features_df.shape[1]

feature_names = [
    "txId",
    "time_step"
]

feature_names += [
    f"feature_{i}"
    for i in range(1, num_columns - 1)
]

features_df.columns = feature_names

In [7]:
print(features_df.columns)
print("Number of ML features:", len(features_df.columns) - 2)

Index(['txId', 'time_step', 'feature_1', 'feature_2', 'feature_3', 'feature_4',
       'feature_5', 'feature_6', 'feature_7', 'feature_8',
       ...
       'feature_156', 'feature_157', 'feature_158', 'feature_159',
       'feature_160', 'feature_161', 'feature_162', 'feature_163',
       'feature_164', 'feature_165'],
      dtype='object', length=167)
Number of ML features: 165


In [8]:
classes_df["class"].value_counts()

class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64

In [9]:
label_mapping = {
    "1": 1,
    "2": 0,
    "unknown": -1
}

classes_df["label"] = (
    classes_df["class"]
    .astype(str)
    .map(label_mapping)
)

In [10]:
classes_df["label"].value_counts()

label
-1    157205
 0     42019
 1      4545
Name: count, dtype: int64

In [11]:
data_df = features_df.merge(
    classes_df[["txId", "label"]],
    on="txId",
    how="left"
)

In [12]:
print(data_df.shape)
data_df.head()

(203769, 168)


,txId,time_step,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165,label
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,-1
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,-1
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792,-1
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792,0
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117,-1


In [13]:
print(
    "Missing labels:",
    data_df["label"].isna().sum()
)

Missing labels: 0


In [14]:
time_distribution = pd.crosstab(
    data_df["time_step"],
    data_df["label"]
)

time_distribution.columns = [
    "unknown" if col == -1
    else "licit" if col == 0
    else "illicit"
    for col in time_distribution.columns
]

time_distribution

,unknown,licit,illicit
time_step,,,
1,5733,2130,17
2,3427,1099,18
3,5342,1268,11
4,4253,1410,30
5,4921,1874,8
6,3843,480,5
7,4845,1101,102
8,3292,1098,67
9,4218,530,248


# train/validation/test split
Train       → time steps 1–34
Validation  → time steps 35–41
Test        → time steps 42–49

In [15]:
train_mask_np = (
    (data_df["time_step"] <= 34)
    &
    (data_df["label"] != -1)
).to_numpy()

val_mask_np = (
    (data_df["time_step"] >= 35)
    &
    (data_df["time_step"] <= 41)
    &
    (data_df["label"] != -1)
).to_numpy()

test_mask_np = (
    (data_df["time_step"] >= 42)
    &
    (data_df["time_step"] <= 49)
    &
    (data_df["label"] != -1)
).to_numpy()

In [16]:
print("Train labeled nodes:", train_mask_np.sum())
print("Validation labeled nodes:", val_mask_np.sum())
print("Test labeled nodes:", test_mask_np.sum())

Train labeled nodes: 29894
Validation labeled nodes: 7829
Test labeled nodes: 8841


In [17]:
np.unique(train_mask_np, return_counts=True)

(array([False,  True]), array([173875,  29894]))

In [18]:
def show_split_distribution(df, mask, split_name):
    
    split_labels = df.loc[mask, "label"]
    
    counts = split_labels.value_counts().sort_index()
    
    print(f"\n{split_name}")
    print("-" * 30)
    
    print("Licit   :", counts.get(0, 0))
    print("Illicit :", counts.get(1, 0))
    
    illicit_ratio = (
        counts.get(1, 0)
        /
        len(split_labels)
    )
    
    print(
        "Illicit percentage:",
        round(illicit_ratio * 100, 2),
        "%"
    )

In [19]:
show_split_distribution(
    data_df,
    train_mask_np,
    "TRAIN"
)

show_split_distribution(
    data_df,
    val_mask_np,
    "VALIDATION"
)

show_split_distribution(
    data_df,
    test_mask_np,
    "TEST"
)


TRAIN
------------------------------
Licit   : 26432
Illicit : 3462
Illicit percentage: 11.58 %

VALIDATION
------------------------------
Licit   : 7154
Illicit : 675
Illicit percentage: 8.62 %

TEST
------------------------------
Licit   : 8433
Illicit : 408
Illicit percentage: 4.61 %


# create the node feature matrix

In [20]:
# We should not feed these into the model:
# txId
# time_step
# label

In [21]:
feature_columns = [
    column
    for column in data_df.columns
    if column.startswith("feature_")
]

In [22]:
print(
    "Number of model features:",
    len(feature_columns)
)

Number of model features: 165


In [23]:
X = data_df[feature_columns].copy()

In [24]:
# Fit scaler using training-period nodes
training_period_mask = (
    data_df["time_step"] <= 34
)

scaler = StandardScaler()

scaler.fit(
    X.loc[training_period_mask]
)

X_scaled = scaler.transform(X)

print(X_scaled.shape)

(203769, 165)


# Convert labels to NumPy

In [25]:
y = data_df["label"].to_numpy()

In [26]:
print(np.unique(y, return_counts=True))

(array([-1,  0,  1]), array([157205,  42019,   4545]))


# Create transaction-ID → node-index mapping

In [27]:
txid_to_index = {
    txid: index
    for index, txid
    in enumerate(data_df["txId"])
}

In [28]:
print(
    "Number of mapped transactions:",
    len(txid_to_index)
)

Number of mapped transactions: 203769


In [29]:
list(txid_to_index.items())[:5]

[(230425980, 0), (5530458, 1), (232022460, 2), (232438397, 3), (230460314, 4)]

In [30]:
edges_df.head()

,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206


In [31]:
print(edges_df.columns)

Index(['txId1', 'txId2'], dtype='object')


In [32]:
edges_df["source"] = (
    edges_df["txId1"]
    .map(txid_to_index)
)

edges_df["target"] = (
    edges_df["txId2"]
    .map(txid_to_index)
)

In [33]:
print(
    "Missing source nodes:",
    edges_df["source"].isna().sum()
)

print(
    "Missing target nodes:",
    edges_df["target"].isna().sum()
)

Missing source nodes: 0
Missing target nodes: 0


In [34]:
edges_df[
    ["source", "target"]
].head()

,source,target
0,0,1
1,2,3
2,4,5
3,6,7
4,8,9


# Create edge_index

In [35]:
edge_index_np = edges_df[
    ["source", "target"]
].to_numpy().T

In [36]:
print(edge_index_np.shape)

(2, 234355)


In [37]:
x_tensor = torch.tensor(
    X_scaled,
    dtype=torch.float32
)

y_tensor = torch.tensor(
    y,
    dtype=torch.long
)

edge_index_tensor = torch.tensor(
    edge_index_np,
    dtype=torch.long
)

train_mask = torch.tensor(
    train_mask_np,
    dtype=torch.bool
)

val_mask = torch.tensor(
    val_mask_np,
    dtype=torch.bool
)

test_mask = torch.tensor(
    test_mask_np,
    dtype=torch.bool
)

In [38]:
print("x:", x_tensor.shape)
print("y:", y_tensor.shape)

print(
    "edge_index:",
    edge_index_tensor.shape
)

print(
    "train_mask:",
    train_mask.shape
)

x: torch.Size([203769, 165])
y: torch.Size([203769])
edge_index: torch.Size([2, 234355])
train_mask: torch.Size([203769])


# Class weights

In [39]:
train_labels = y[
    train_mask_np
]

In [40]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_labels
)

In [41]:
print(
    "Class weights:",
    class_weights
)

Class weights: [0.5654888  4.31744656]


In [42]:
class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
)

# Create PyTorch Geometric Data object

In [44]:
from torch_geometric.data import Data

C:\Users\nakul\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [45]:
graph_data = Data(
    x=x_tensor,
    edge_index=edge_index_tensor,
    y=y_tensor,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask
)

In [46]:
graph_data

Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], train_mask=[203769], val_mask=[203769], test_mask=[203769])

In [47]:
# Important graph decision: directed or undirected?
# Elliptic edges represent transaction flow:
# A → B

# So technically the graph is directed.
# However, the standard GCNConv implementation generally works well when message passing includes both directions.
# For our simple V1, I recommend making the graph undirected.

In [48]:
from torch_geometric.utils import to_undirected

graph_data.edge_index = to_undirected(
    graph_data.edge_index
)

In [49]:
print(
    graph_data.edge_index.shape
)

torch.Size([2, 468710])


In [50]:
# The number of edges will increase because:
# A → B

# becomes effectively:
# A → B
# B → A

# For a first vanilla GCN, this is simpler and usually appropriate.
# We should document this preprocessing decision in the report.

In [51]:
print("Number of nodes:")
print(graph_data.num_nodes)

print("\nNumber of edges:")
print(graph_data.num_edges)

print("\nNumber of features:")
print(graph_data.num_node_features)

print("\nTraining nodes:")
print(graph_data.train_mask.sum().item())

print("\nValidation nodes:")
print(graph_data.val_mask.sum().item())

print("\nTesting nodes:")
print(graph_data.test_mask.sum().item())

Number of nodes:
203769

Number of edges:
468710

Number of features:
165

Training nodes:
29894

Validation nodes:
7829

Testing nodes:
8841


In [52]:
print(
    "Unique labels:",
    torch.unique(
        graph_data.y,
        return_counts=True
    )
)

Unique labels: (tensor([-1,  0,  1]), tensor([157205,  42019,   4545]))


# Save the processed graph

In [53]:
OUTPUT_PATH = "../data/processed/elliptic_graph.pt"

In [54]:
from pathlib import Path

Path(
    "../data/processed"
).mkdir(
    parents=True,
    exist_ok=True
)

In [55]:
torch.save(
    graph_data,
    OUTPUT_PATH
)

In [56]:
torch.save(
    class_weights_tensor,
    "../data/processed/class_weights.pt"
)

In [57]:
import joblib

joblib.dump(
    scaler,
    "../data/processed/scaler.pkl"
)

['../data/processed/scaler.pkl']